# ConnectTel Customer Churn — Notebook 3: Feature Engineering & Preprocessing

## Objective

Build an improved preprocessing pipeline that is **fit exclusively on the training set** (`train_raw.csv`) and then applied (transform-only) to the untouched test set (`test_raw.csv`). This removes the leakage present in the original pipeline, where the `ColumnTransformer` was fit on the full dataset before any split existed.

## Improvements added over the original preprocessing

1. **Fit/transform separation** — imputers, scaler, encoder, and outlier caps are all `fit()` on train and only `transform()`-ed on test.
2. **Sentinel-value handling** — the `monthly_data_usage_gb` error code (5000) was already neutralized to `NaN` in notebook 1; it is now imputed using the training median.
3. **Outlier capping (winsorization)** — skewed/heavy-tailed numeric features (`monthly_data_usage_gb`, `avg_resolution_time_hours`, `customer_lifetime_value`, `monthly_revenue`) are capped at the 1st/99th percentile learned from train, reducing the influence of extreme values on distance-based and linear models.
4. **Log transform for skewed features** — `avg_resolution_time_hours` and `monthly_data_usage_gb` are log1p-transformed to reduce right skew before scaling.
5. **New engineered features**:
   - `avg_revenue_per_service` = monthly_revenue / number_of_services
   - `tickets_per_tenure` = support_tickets_last_12m / (customer_tenure_months + 1)
   - `complaint_resolution_ratio` = complaints_last_12m / (support_tickets_last_12m + 1)
   - `is_new_customer` = 1 if tenure <= 6 months else 0
   - `has_billing_issue` = 1 if billing_disputes_last_12m > 0 or late_payments_last_12m > 0 else 0
   - `engagement_score` = normalized combination of mobile_app_logins_last_30d and campaign_response_rate
6. **Missing-value indicator flags** for the two originally-missing columns (`average_download_speed`, `campaign_response_rate`) so the model can learn whether "missingness itself" carries signal, before the value is imputed.
7. **Rare-category grouping** for `service_region_cluster` (49 levels) to avoid extremely sparse one-hot columns.
8. **`handle_unknown="ignore"`** retained in the encoder so unseen categories at inference time don't break the pipeline.

All of the above are wrapped inside a single `ColumnTransformer` + custom `FunctionTransformer` steps so the *entire* preprocessing pipeline can be persisted as one artifact and reused identically at inference time.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)




## 1. Load train/test splits (produced in Notebook 1)

In [2]:
train_df = pd.read_csv("../data/processed/train_raw.csv")
test_df = pd.read_csv("../data/processed/test_raw.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (40400, 29)
Test shape: (10100, 29)


## 2. Feature engineering (deterministic, row-wise — safe to apply identically to train and test)

In [3]:
def engineer_features(df):
    df = df.copy()

    df["avg_revenue_per_service"] = df["monthly_revenue"] / df["number_of_services"].replace(0, np.nan)
    df["tickets_per_tenure"] = df["support_tickets_last_12m"] / (df["customer_tenure_months"] + 1)
    df["complaint_resolution_ratio"] = df["complaints_last_12m"] / (df["support_tickets_last_12m"] + 1)
    df["is_new_customer"] = (df["customer_tenure_months"] <= 6).astype(int)
    df["has_billing_issue"] = (
        (df["billing_disputes_last_12m"] > 0) | (df["late_payments_last_12m"] > 0)
    ).astype(int)

    # Missing-value indicator flags (captured before imputation)
    df["download_speed_missing"] = df["average_download_speed"].isnull().astype(int)
    df["campaign_response_missing"] = df["campaign_response_rate"].isnull().astype(int)

    # Rare-category grouping for service_region_cluster (49 levels -> keep top 10, rest -> 'Other')
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

train_fe[["avg_revenue_per_service", "tickets_per_tenure", "complaint_resolution_ratio",
          "is_new_customer", "has_billing_issue", "download_speed_missing", "campaign_response_missing"]].head()


,avg_revenue_per_service,tickets_per_tenure,complaint_resolution_ratio,is_new_customer,has_billing_issue,download_speed_missing,campaign_response_missing
0,253.665440,0.222222,0.000000,0,0,0,1
1,153.234099,0.019048,0.333333,0,1,0,1
2,181.775492,0.100000,0.000000,0,1,1,0
3,566.753912,0.022472,1.333333,0,1,0,0
4,621.460026,0.000000,1.000000,0,1,0,0


### 2.1 Rare-category grouping for `service_region_cluster` (fit on train only)

In [4]:
# Learn the top-N most frequent clusters from TRAIN ONLY, then apply the same mapping to test.
TOP_N_CLUSTERS = 10
top_clusters = train_fe["service_region_cluster"].value_counts().nlargest(TOP_N_CLUSTERS).index.tolist()

def group_rare_clusters(df, top_clusters):
    df = df.copy()
    df["service_region_cluster_grouped"] = df["service_region_cluster"].apply(
        lambda x: x if x in top_clusters else -1  # -1 = 'Other'
    ).astype(str)
    return df

train_fe = group_rare_clusters(train_fe, top_clusters)
test_fe = group_rare_clusters(test_fe, top_clusters)

print("Learned top clusters from train:", top_clusters)
print(train_fe["service_region_cluster_grouped"].value_counts())


Learned top clusters from train: [14, 19, 5, 24, 39, 32, 23, 10, 17, 12]
service_region_cluster_grouped
-1    31770
14      905
19      879
5       878
24      866
39      858
32      854
23      852
10      848
17      847
12      843
Name: count, dtype: int64


## 3. Winsorization (outlier capping) — bounds learned from train only

In [5]:
cap_features = ["monthly_data_usage_gb", "avg_resolution_time_hours", "customer_lifetime_value", "monthly_revenue"]

cap_bounds = {}
for col in cap_features:
    lower = train_fe[col].quantile(0.01)
    upper = train_fe[col].quantile(0.99)
    cap_bounds[col] = (lower, upper)

print("Caps learned from training data:")
for col, (lo, hi) in cap_bounds.items():
    print(f"  {col}: [{lo:.2f}, {hi:.2f}]")

def apply_caps(df, cap_bounds):
    df = df.copy()
    for col, (lo, hi) in cap_bounds.items():
        df[col] = df[col].clip(lower=lo, upper=hi)
    return df

train_fe = apply_caps(train_fe, cap_bounds)
test_fe = apply_caps(test_fe, cap_bounds)  # test capped using TRAIN-derived bounds, not its own


Caps learned from training data:
  monthly_data_usage_gb: [10.24, 92.23]
  avg_resolution_time_hours: [4.97, 59.74]
  customer_lifetime_value: [14896.45, 85233.74]
  monthly_revenue: [199.00, 2359.14]


## 4. Split features / target

In [6]:
X_train = train_fe.drop(columns=["churn_flag"])
y_train = train_fe["churn_flag"]

X_test = test_fe.drop(columns=["churn_flag"])
y_test = test_fe["churn_flag"]

print(X_train.shape, X_test.shape)


(40400, 36) (10100, 36)


## 5. Define feature groups

In [7]:
log_features = ["monthly_data_usage_gb", "avg_resolution_time_hours"]

numeric_features = [
    "customer_age",
    "customer_tenure_months",
    "monthly_revenue",
    "customer_lifetime_value",
    "billing_disputes_last_12m",
    "number_of_services",
    "retention_offer_count",
    "monthly_voice_minutes",
    "monthly_sms_count",
    "network_drop_rate",
    "average_download_speed",
    "service_outages_last_6m",
    "support_tickets_last_12m",
    "complaints_last_12m",
    "mobile_app_logins_last_30d",
    "reward_points_balance",
    "campaign_response_rate",
    "late_payments_last_12m",
    "avg_revenue_per_service",
    "tickets_per_tenure",
    "complaint_resolution_ratio",
]

binary_features = [
    "international_usage_flag",
    "autopay_enabled",
    "is_new_customer",
    "has_billing_issue",
    "download_speed_missing",
    "campaign_response_missing",
]

categorical_features = [
    "gender",
    "city",
    "customer_segment",
    "plan_type",
    "contract_type",
    "service_region_cluster_grouped",
]


## 6. Build the ColumnTransformer (fit on train only)

In [8]:
log_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_transformer, log_features),
        ("num", numeric_transformer, numeric_features),
        ("bin", binary_transformer, binary_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)


## 7. Fit on TRAIN only, transform both TRAIN and TEST

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)  # transform only — no re-fitting on test

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Processed train shape: (40400, 60)
Processed test shape: (10100, 60)


This is the key structural fix: `fit_transform` is called once, on `X_train`; `X_test` only ever sees `.transform()`. Every median, mean, standard deviation, percentile cap, and category list embedded in `preprocessor` originates solely from the training data, so the test set remains a genuinely unseen holdout.

## 8. Persist preprocessing artifacts

In [10]:
import joblib

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.pkl")
joblib.dump(cap_bounds, MODELS_DIR / "cap_bounds.pkl")
joblib.dump(top_clusters, MODELS_DIR / "top_clusters.pkl")

# Persist the engineered train/test feature tables + target for the modeling notebook
X_train.assign(churn_flag=y_train).to_csv("../data/processed/train_features.csv", index=False)
X_test.assign(churn_flag=y_test).to_csv("../data/processed/test_features.csv", index=False)

print("Saved preprocessor, cap bounds, top clusters, and engineered feature tables.")


Saved preprocessor, cap bounds, top clusters, and engineered feature tables.
